# Gold Layer - Dimensional Model

## Schema
- DimDate: shared across both facts
- DimRegion: regional unemployment breakdown
- FactUnemployment: monthly unemployment rate by region
- FactMacroIndicators: monthly CPI + SNB policy rate (national)

## Design decisions
- Two fact tables: different geographic grain (regional vs national)
- Linked via DimDate in Power BI for cross-indicator analysis
- Surrogate keys on all dimensions

In [1]:
import pandas as pd
from pathlib import Path

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold")
GOLD_PATH.mkdir(parents=True, exist_ok=True)

# Load silver
df_unemployment = pd.read_parquet(SILVER_PATH / "silver_unemployment.parquet")
df_cpi = pd.read_parquet(SILVER_PATH / "silver_cpi.parquet")
df_policy = pd.read_parquet(SILVER_PATH / "silver_policy_rate.parquet")

print(f"Silver unemployment : {df_unemployment.shape}")
print(f"Silver CPI          : {df_cpi.shape}")
print(f"Silver policy rate  : {df_policy.shape}")

Silver unemployment : (1365, 9)
Silver CPI          : (394, 7)
Silver policy rate  : (197, 7)


In [2]:
# ── DimDate ──────────────────────────────────────────────────────────────────

def build_dim_date(periods: pd.Series) -> pd.DataFrame:
    """
    Build DimDate from a series of YYYY-MM period strings.
    Surrogate key: integer YYYYMM (e.g. 202301)
    """
    unique_periods = periods.drop_duplicates().sort_values().reset_index(drop=True)
    
    dim_date = pd.DataFrame({"period": unique_periods})
    dim_date["date_id"]  = dim_date["period"].str.replace("-", "").astype(int)
    dim_date["year"]     = dim_date["period"].str[:4].astype(int)
    dim_date["month"]    = dim_date["period"].str[5:7].astype(int)
    dim_date["quarter"]  = ((dim_date["month"] - 1) // 3 + 1).astype(int)
    dim_date["year_month_label"] = dim_date["period"]  # for Power BI axis labels
    
    return dim_date[["date_id", "period", "year", "month", "quarter", "year_month_label"]]


# Combine all periods from all sources
all_periods = pd.concat([
    df_unemployment["period"],
    df_cpi["period"],
    df_policy["period"]
])

dim_date = build_dim_date(all_periods)

print(f"DimDate : {len(dim_date)} rows")
print(f"\nSample:\n{dim_date.head(4)}")
print(f"\nRange: {dim_date['period'].min()} → {dim_date['period'].max()}")

DimDate : 197 rows

Sample:
   date_id   period  year  month  quarter year_month_label
0   201001  2010-01  2010      1        1          2010-01
1   201002  2010-02  2010      2        1          2010-02
2   201003  2010-03  2010      3        1          2010-03
3   201004  2010-04  2010      4        2          2010-04

Range: 2010-01 → 2026-05


In [8]:
def build_dim_region(df: pd.DataFrame) -> pd.DataFrame:
    region_meta = {
        "Région lémanique"    : (46.3167, 6.5833, "Geneva, Lausanne, Sion"),
        "Espace Mittelland"   : (46.9833, 7.2167, "Bern, Fribourg, Neuchâtel"),
        "Suisse du Nord-Ouest": (47.5583, 7.5729, "Basel, Aarau, Soleure"),
        "Zurich"              : (47.3769, 8.5417, "Zurich, Winterthur"),
        "Suisse orientale"    : (47.4245, 9.3767, "St-Gallen, Chur, Schaffhausen"),
        "Suisse centrale"     : (47.0502, 8.3093, "Lucerne, Zug, Altdorf"),
        "Tessin"              : (46.0037, 8.9511, "Lugano, Bellinzona, Locarno"),
    }
    unique_regions = df["region"].drop_duplicates().sort_values().reset_index(drop=True)
    dim_region = pd.DataFrame({"region_name": unique_regions})
    dim_region["region_id"]   = range(1, len(dim_region) + 1)
    dim_region["latitude"]    = dim_region["region_name"].map(lambda x: region_meta[x][0])
    dim_region["longitude"]   = dim_region["region_name"].map(lambda x: region_meta[x][1])
    dim_region["main_cities"] = dim_region["region_name"].map(lambda x: region_meta[x][2])
    return dim_region[["region_id", "region_name", "latitude", "longitude", "main_cities"]]

dim_region = build_dim_region(df_unemployment)
print(dim_region)

   region_id           region_name  latitude  longitude  \
0          1     Espace Mittelland   46.9833     7.2167   
1          2      Région lémanique   46.3167     6.5833   
2          3       Suisse centrale   47.0502     8.3093   
3          4  Suisse du Nord-Ouest   47.5583     7.5729   
4          5      Suisse orientale   47.4245     9.3767   
5          6                Tessin   46.0037     8.9511   
6          7                Zurich   47.3769     8.5417   

                     main_cities  
0      Bern, Fribourg, Neuchâtel  
1         Geneva, Lausanne, Sion  
2          Lucerne, Zug, Altdorf  
3          Basel, Aarau, Soleure  
4  St-Gallen, Chur, Schaffhausen  
5    Lugano, Bellinzona, Locarno  
6             Zurich, Winterthur  


In [9]:
def build_fact_unemployment(df, dim_date, dim_region):
    fact = df.copy()
    fact = fact.merge(dim_date[["period", "date_id"]], on="period", how="left")
    fact = fact.merge(dim_region[["region_name", "region_id"]], left_on="region", right_on="region_name", how="left")
    return fact[["date_id", "region_id", "unemployment_rate", "_run_id", "_ingested_at"]]

fact_unemployment = build_fact_unemployment(df_unemployment, dim_date, dim_region)
print(f"FactUnemployment : {len(fact_unemployment)} rows")
print(f"Null keys: {fact_unemployment[['date_id','region_id']].isna().sum().to_dict()}")

FactUnemployment : 1365 rows
Null keys: {'date_id': 0, 'region_id': 0}


In [10]:
def build_fact_macro(df_cpi, df_policy, dim_date):
    cpi_pivot = df_cpi.pivot(index="period", columns="series_key", values="cpi_value").reset_index()
    cpi_pivot = cpi_pivot.rename(columns={"LD2010100": "cpi_index", "VVP": "cpi_yoy"})
    fact = cpi_pivot.merge(df_policy[["period", "swiss_policy_rate", "_rate_source"]], on="period", how="inner")
    fact = fact.merge(dim_date[["period", "date_id"]], on="period", how="left")
    return fact[["date_id", "period", "cpi_index", "cpi_yoy", "swiss_policy_rate", "_rate_source"]].sort_values("period").reset_index(drop=True)

fact_macro = build_fact_macro(df_cpi, df_policy, dim_date)
print(f"FactMacroIndicators : {len(fact_macro)} rows")
print(f"Null keys: {fact_macro[['date_id','cpi_index','swiss_policy_rate']].isna().sum().to_dict()}")

FactMacroIndicators : 197 rows
Null keys: {'date_id': 0, 'cpi_index': 0, 'swiss_policy_rate': 0}


In [11]:
dim_date.to_parquet(GOLD_PATH / "dim_date.parquet", index=False)
dim_region.to_parquet(GOLD_PATH / "dim_region.parquet", index=False)
fact_unemployment.to_parquet(GOLD_PATH / "fact_unemployment.parquet", index=False)
fact_macro.to_parquet(GOLD_PATH / "fact_macro_indicators.parquet", index=False)

print("Gold layer saved:")
print(f"  dim_date              : {len(dim_date)} rows")
print(f"  dim_region            : {len(dim_region)} rows")
print(f"  fact_unemployment     : {len(fact_unemployment)} rows")
print(f"  fact_macro_indicators : {len(fact_macro)} rows")

Gold layer saved:
  dim_date              : 197 rows
  dim_region            : 7 rows
  fact_unemployment     : 1365 rows
  fact_macro_indicators : 197 rows
